# 02 -- TCN Encoder From Scratch

**Author**: Tamer Atesyakar

We implement a tiny Temporal Convolutional Network (TCN) using dilated causal 1-D convolutions, attach an NT-Xent contrastive head, and contrast its receptive field and performance against an LSTM on the same synthetic sequence.

**Citations**
- Bai, Kolter & Koltun (2018). *An Empirical Evaluation of Generic Convolutional and Recurrent Networks for Sequence Modeling.* arXiv:1803.01271.
- Chen, Kornblith et al. (2020). *A Simple Framework for Contrastive Learning of Visual Representations (NT-Xent).* ICML.
- Hochreiter & Schmidhuber (1997). *Long Short-Term Memory.* Neural Computation.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_OK = True
except ImportError as e:
    TORCH_OK = False
    print(f'torch not available: {e}')

C_COLD = '#0f3460'
C_HOT  = '#e94560'
plt.rcParams.update({'axes.grid': True, 'grid.alpha': 0.25})

## 2.1 Receptive-field math

For a causal TCN stack of `L` layers with kernel size `k` and exponentially growing dilation `d_l = 2^l`, the total receptive field is

$$R = 1 + (k - 1) \cdot \sum_{l=0}^{L-1} 2^l = 1 + (k-1)(2^L - 1).$$

For `k=3, L=4` this gives `R = 1 + 2 * 15 = 31` past timesteps, enough for the last 30 keystrokes in a typing sample.

In [ ]:
def receptive_field(k, L):
    return 1 + (k - 1) * (2**L - 1)

fig, ax = plt.subplots(figsize=(8, 3.6))
Ls = np.arange(1, 9)
for k, color in [(2, C_COLD), (3, C_HOT)]:
    ax.plot(Ls, [receptive_field(k, L) for L in Ls],
            marker='o', color=color, label=f'kernel={k}')
ax.set_xlabel('depth L')
ax.set_ylabel('receptive field (timesteps)')
ax.set_title('Dilated causal conv stack: receptive field grows 2^L')
ax.legend()
plt.tight_layout(); plt.show()

## 2.2 Dilated causal conv block

In [ ]:
if TORCH_OK:
    class CausalConv1d(nn.Module):
        def __init__(self, cin, cout, k, dilation):
            super().__init__()
            self.pad = (k - 1) * dilation
            self.conv = nn.Conv1d(cin, cout, k, dilation=dilation)

        def forward(self, x):
            # pad only on the left -> strictly causal
            x = F.pad(x, (self.pad, 0))
            return self.conv(x)

    class TCNBlock(nn.Module):
        def __init__(self, c, k, dilation, dropout=0.1):
            super().__init__()
            self.c1 = CausalConv1d(c, c, k, dilation)
            self.c2 = CausalConv1d(c, c, k, dilation)
            self.drop = nn.Dropout(dropout)

        def forward(self, x):
            r = x
            x = F.relu(self.c1(x)); x = self.drop(x)
            x = F.relu(self.c2(x)); x = self.drop(x)
            return x + r

    class TinyTCN(nn.Module):
        def __init__(self, din, c=32, k=3, L=4, proj=16):
            super().__init__()
            self.stem = nn.Conv1d(din, c, 1)
            self.blocks = nn.ModuleList(
                [TCNBlock(c, k, dilation=2**l) for l in range(L)]
            )
            self.head = nn.Linear(c, proj)

        def forward(self, x):      # x: (B, T, D)
            h = self.stem(x.transpose(1, 2))
            for b in self.blocks:
                h = b(h)
            h = h.mean(dim=-1)      # global temporal avg
            z = self.head(h)
            return F.normalize(z, dim=-1)

    m = TinyTCN(din=32)
    n_params = sum(p.numel() for p in m.parameters())
    print(f'TinyTCN parameters: {n_params:,}')
else:
    print('torch unavailable -- model definition skipped')

## 2.3 NT-Xent contrastive loss

For a batch of `2N` projections (two augmentations per anchor), NT-Xent is

$$\mathcal{L}_{i,j} = -\log\frac{\exp(\mathrm{sim}(z_i,z_j)/\tau)}{\sum_{k\neq i}\exp(\mathrm{sim}(z_i,z_k)/\tau)}.$$

We implement it below (Chen et al. 2020).

In [ ]:
if TORCH_OK:
    def nt_xent(z1, z2, tau=0.1):
        z = torch.cat([z1, z2], dim=0)              # (2N, d)
        sim = z @ z.t() / tau                        # cosine because z is l2-normed
        N = z1.shape[0]
        mask = torch.eye(2 * N, device=z.device).bool()
        sim.masked_fill_(mask, -1e9)
        labels = torch.cat([torch.arange(N, 2*N), torch.arange(0, N)]).to(z.device)
        return F.cross_entropy(sim, labels)

    # Sanity: random tensors produce a well-conditioned loss.
    torch.manual_seed(0)
    z1 = F.normalize(torch.randn(8, 16), dim=-1)
    z2 = F.normalize(torch.randn(8, 16), dim=-1)
    print(f'NT-Xent on random batch (tau=0.1): {nt_xent(z1, z2).item():.3f}')

## 2.4 LSTM baseline on the same task

In [ ]:
if TORCH_OK:
    class TinyLSTM(nn.Module):
        def __init__(self, din, h=32, proj=16):
            super().__init__()
            self.lstm = nn.LSTM(din, h, batch_first=True)
            self.head = nn.Linear(h, proj)

        def forward(self, x):
            y, _ = self.lstm(x)
            z = self.head(y[:, -1])
            return F.normalize(z, dim=-1)

    lstm = TinyLSTM(din=32)
    print(f'TinyLSTM parameters: {sum(p.numel() for p in lstm.parameters()):,}')

## 2.5 Toy training: distinguish two typing rhythms

We build two clusters of synthetic 32-dim sequences (relaxed vs stressed) and show that the TCN separates them after a handful of gradient steps.

In [ ]:
if TORCH_OK:
    def synth_seq(kind, T=32):
        base = np.zeros((T, 32), dtype=np.float32)
        if kind == 'relaxed':
            base[:, 0] = np.random.normal(120, 15, T)    # mean IKI
            base[:, 1] = np.random.normal(25, 4, T)
        else:  # stressed
            base[:, 0] = np.random.normal(90, 35, T)
            base[:, 1] = np.random.normal(55, 9, T)
        base += np.random.normal(0, 0.02, base.shape)
        return base

    def batch(n=16):
        xs = [synth_seq('relaxed') for _ in range(n)] + [synth_seq('stressed') for _ in range(n)]
        return torch.tensor(np.stack(xs), dtype=torch.float32)

    torch.manual_seed(1)
    m = TinyTCN(din=32)
    opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    losses = []
    for step in range(80):
        x = batch()
        # Simple augmentation: additive noise
        x1 = x + 0.01 * torch.randn_like(x)
        x2 = x + 0.01 * torch.randn_like(x)
        z1, z2 = m(x1), m(x2)
        loss = nt_xent(z1, z2, tau=0.2)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(losses, color=C_HOT)
    ax.set_xlabel('step'); ax.set_ylabel('NT-Xent loss')
    ax.set_title('Contrastive objective decreases on toy typing rhythms')
    plt.tight_layout(); plt.show()

## 2.6 Embedding geometry

In [ ]:
if TORCH_OK:
    with torch.no_grad():
        x = batch(32)
        z = m(x).numpy()
    # Since proj=16 and l2-normed, project onto the first two PCA components.
    zc = z - z.mean(0, keepdims=True)
    U, S, Vt = np.linalg.svd(zc, full_matrices=False)
    p = zc @ Vt[:2].T
    n = x.shape[0] // 2
    fig, ax = plt.subplots(figsize=(5.2, 5))
    ax.scatter(p[:n, 0], p[:n, 1], color=C_COLD, label='relaxed', s=40, alpha=0.85)
    ax.scatter(p[n:, 0], p[n:, 1], color=C_HOT,  label='stressed', s=40, alpha=0.85)
    ax.set_title('TCN embedding -- first 2 PCA components')
    ax.legend(); ax.set_aspect('equal')
    plt.tight_layout(); plt.show()

## 2.7 TCN vs LSTM: summary

- **Parallelism**: TCN forward pass parallelises across time; LSTM is sequential.
- **Receptive field**: TCN grows 2^L; LSTM is effectively limited by gradient flow over long ranges.
- **Stability**: residual connections + weight-norm make TCNs easier to train (Bai et al. 2018).
- **Memory**: for T=32 and modest depth the TCN is cheaper on modern accelerators.